In [4]:
import torch 
import math 


# Attention
## Single head attention 


In [5]:
def single_head_attention(X, WQ, WK, WV, causal=True):
    """
    X: (N, d)
    WQ, WK: (d, d_k)
    WV: (d, d_v)

    Returns:
        A: attention outputs, shape (N, d_v)
        attn_weights: self-attention distribution, shape (N, N)
        Q, K, V: query, key, value matrices, shape (N, d_k)
    """
    Q = X @ WQ  # (N, d_k)
    K = X @ WK  # (N, d_k)
    V = X @ WV  # (N, d_v)

    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if causal:
        N = scores.size(-1)
        mask = torch.triu(
            torch.ones(N, N, dtype=bool, device=scores.device), 
            diagonal=1, 
        )
        scores = scores.masked_fill(mask, float("-inf"))
    
    attn_weights = torch.softmax(scores, dim=-1)  # (N, N)
    A = attn_weights @ V  # (N, d_v)

    return A, attn_weights, Q, K, V



In [6]:
torch.manual_seed(42)
N, d, d_k, d_v = 4, 8, 8, 8

X = torch.randn(N, d)
WQ = torch.randn(d, d_k)
WK = torch.randn(d, d_k)
WV = torch.randn(d, d_v)

A, attn_weights, Q, K, V = single_head_attention(X, WQ, WK, WV)


print(f"A shape: {A.shape}")
print(f"attn_weights shape: {attn_weights.shape}")
print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")



A shape: torch.Size([4, 8])
attn_weights shape: torch.Size([4, 4])
Q shape: torch.Size([4, 8])
K shape: torch.Size([4, 8])
V shape: torch.Size([4, 8])


## Multi-head attention

Note: we call our single_head_attention function due to pedagocical consideration and not implementation efficiency. 



In [7]:
def multi_head_attention_v1(X, WQ_list, WK_list, WV_list, WO, causal=True):
    """
    X: (N, d_model)
    WO: (num_heads*d_v, d_model)
    """
    head_outputs = []
    for WQ, WK, WV in zip(WQ_list, WK_list, WV_list):
        A, _, _, _, _ = single_head_attention(X, WQ, WK, WV, causal=causal)
        head_outputs.append(A)
    
    concat_head = torch.cat(head_outputs, dim=-1)  # (N, h*d_v)
    output = concat_head @ WO  # (N, d_model)
    return output, head_outputs




### Residual stream


In [2]:
import torch.nn.functional as F 

def attention_residual_block_v1(X, WQ_list, WK_list, WV_list, WO, causal=True):
    """
    X: residual stream, shape (N, d)

    Returns:
        t1: layer-normalized residual stream
        t2: multi-head attention output
        t3: updated residual stream after residual connection
    """
    d = X.size(-1)
    t1 = F.layer_norm(X, normalized_shape=(d,), eps=1e-8)
    t2, heads_outputs = multi_head_attention_v1(
        t1, WQ_list, WK_list, WV_list, WO, causal=causal
    )
    t3 = X + t2  # residual connection
    return t1, t2, t3, heads_outputs
    

In [8]:
torch.manual_seed(42)

N = 4
d_model = 8
num_heads = 2
d_k = 4
d_v = 4

X = torch.randn(N, d_model)

WQ_list = [torch.randn(d_model, d_k) for _ in range(num_heads)]
WK_list = [torch.randn(d_model, d_k) for _ in range(num_heads)]
WV_list = [torch.randn(d_model, d_v) for _ in range(num_heads)]
WO = torch.randn(num_heads * d_v, d_model)

t1, t2, t3, head_outputs = attention_residual_block_v1(
    X, WQ_list, WK_list, WV_list, WO, causal=True
)

print("X shape :", X.shape)
print("t1 shape:", t1.shape)
print("t2 shape:", t2.shape)
print("t3 shape:", t3.shape)
print("num heads:", len(head_outputs))
print("one head shape:", head_outputs[0].shape)

X shape : torch.Size([4, 8])
t1 shape: torch.Size([4, 8])
t2 shape: torch.Size([4, 8])
t3 shape: torch.Size([4, 8])
num heads: 2
one head shape: torch.Size([4, 4])


In [9]:
def feed_forward_network(X, W1, b1, W2, b2):
    """
    X: (N, d)
    W1: (d, d_ff)
    b1: (d_ff)
    W2: (d_ff, d)
    b2: (d)

    Returns:
        output: (N, d)
    """
    hidden = X @ W1 + b1
    hidden = F.relu(hidden)
    output = hidden @ W2 + b2
    return output
    

### Full transformer block 

In [11]:
def transformer_block_v1(X, WQ_list, WK_list, WV_list, WO, W1, b1, W2, b2, causal=True):
    """
    X: residual stream, shape (N, d)
    
    Returns:
        t1: layer-normalized residual stream
        t2: multi-head attention output
        t3: feed-forward network output
        t4: updated residual stream after residual connection
    """
    d = X.size(-1)

    # First layer norm + attention + residual
    t1 = F.layer_norm(X, normalized_shape=(d,), eps=1e-8)
    t2, heads_outputs = multi_head_attention_v1(
        t1, WQ_list, WK_list, WV_list, WO, causal=causal
    )
    t3 = X + t2 

    # Second layer norm + feed-forward + residual
    t4 = F.layer_norm(t3, normalized_shape=(d,))
    t5 = feed_forward_network(t4, W1, b1, W2, b2)
    h = t5 + t3
    
    return t1, t2, t3, t4, t5, h, heads_outputs

In [12]:
torch.manual_seed(42)

N = 4
d_model = 8
num_heads = 2
d_k = 4
d_v = 4
d_ff = 16

X = torch.randn(N, d_model)

WQ_list = [torch.randn(d_model, d_k) for _ in range(num_heads)]
WK_list = [torch.randn(d_model, d_k) for _ in range(num_heads)]
WV_list = [torch.randn(d_model, d_v) for _ in range(num_heads)]
WO = torch.randn(num_heads * d_v, d_model)

W1 = torch.randn(d_model, d_ff)
b1 = torch.randn(d_ff)
W2 = torch.randn(d_ff, d_model)
b2 = torch.randn(d_model)

t1, t2, t3, t4, t5, h, head_outputs = transformer_block_v1(
    X, WQ_list, WK_list, WV_list, WO, W1, b1, W2, b2, causal=True
)

print("X shape :", X.shape)
print("t1 shape:", t1.shape)
print("t2 shape:", t2.shape)
print("t3 shape:", t3.shape)
print("t4 shape:", t4.shape)
print("t5 shape:", t5.shape)
print("h shape :", h.shape)

X shape : torch.Size([4, 8])
t1 shape: torch.Size([4, 8])
t2 shape: torch.Size([4, 8])
t3 shape: torch.Size([4, 8])
t4 shape: torch.Size([4, 8])
t5 shape: torch.Size([4, 8])
h shape : torch.Size([4, 8])


### PyTorch optimized 

Here is optimized, batched Multi-head attention implementaiton, a good practice would modify the coed above using multi_head_attention function for a more elegant and efficient implementation


In [ ]:
from torch._prims_common import FloatLike


def multi_head_attention(X, WQ, WK, WV, WO, causal=True):

    """
    X: (N, d)
    WQ: (h, d, d_k)
    WK: (h, d, d_k)
    WV: (h, d, d_v)
    WO: (h * d_v, d)

    Returns:
        output: (N, d)
        attn_weights: (h, N, N)
        head_outputs: (h, N, d_v)
    """
    h, d, d_k = WQ.shape  # head, model dimension, query length
    N = X.size(0)
    d_v = WV.size(-1)
    
    Q = torch.einsum("nd, hdk -> hnk", X, WQ)
    K = torch.einsum("nd, hdk -> hnk", X, WK)
    V = torch.einsum("nd, hdv -> hnv", X, WV)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if causal:
        mask = torch.triu(
            torch.ones(N, N, dtype=bool, device=scores.device), 
            diagonal=1, 
        )
        scores = scores.masked_fill(mask, float("-inf"))
    
    attn_weights = torch.softmax(scores, dim=-1)   # (h, N, N)
    head_outputs = torch.matmul(attn_weights, V)  # (h, N, d_v)

    # concatenate heads:  (N, h * d_v)
    concat_heads = head_outputs.permute(1, 0, 2).reshape(N, h * d_v)

    # Output projection 
    output = concat_heads @ WO  # (N, d)

    return output, attn_weights, head_outputs

